# Ear-midpoint velocity + smoothing optimization (`pirouette_data.kinematics`)

This notebook computes the **signed** velocity of the mouse from the midpoint between the two ears
(in mm/s) and tunes the Gaussian smoothing `sigma`.

- Velocity is the ear-midpoint position differentiated against `harp_time`.
- The sign is **forward (+) / backward (-)**, from the ear-orthogonal facing direction (the same
  construction as the heading estimate).
- Missing ears: the present ear is used; if both are missing the position is interpolated.
- The instantaneous velocity is smoothed with `scipy.ndimage.gaussian_filter1d`; `sigma` is in
  **samples/frames**.

**Goal of the sweep:** pick the smallest `sigma` that removes frame-to-frame spikiness while
preserving the global velocity structure.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

from pirouette_data import ingestion, kinematics, processing

pd.set_option("display.max_columns", 60)

## Load one hour of pose data with real Harp timestamps

Velocity needs (a) millimetre coordinates (`processing.append_mm_columns`) and (b) a `harp_time`
column in seconds. We load a single `.h5` and pull its matching per-frame Harp time from S3.

In [ ]:
POSE_DIR = r"C:/Users/brandon.pratt/Desktop/data/body-kinematics/pose_data"
S3_VIDEO_URI = "s3://aind-open-data/854393_2026-06-09_19-34-26/behavior-videos"
LIKELIHOOD = 0.6

fname = sorted(Path(POSE_DIR).glob("*.h5"))[0]
df = ingestion.load_pose_h5(fname).reset_index()

camera, timestamp, _ = ingestion.parse_camera_and_timestamp(fname.name)
harp = ingestion.load_harp_seconds(S3_VIDEO_URI, camera, timestamp)
df["harp_time"] = harp[: len(df)]

df = processing.append_mm_columns(df, likelihood_threshold=LIKELIHOOD)
print(f"{fname.name}: {df.shape}")
print(f"duration: {(df['harp_time'].iloc[-1] - df['harp_time'].iloc[0]):.1f} s")

## Instantaneous velocity

In [ ]:
inst, _ = kinematics.ear_velocity_estimate(
    df, likelihood_threshold=LIKELIHOOD, smoothing_sigma=0.0
)
t = df["harp_time"].to_numpy()
t0 = t - t[0]

print(f"median |v|: {np.nanmedian(np.abs(inst)):.1f} mm/s   p99 |v|: {np.nanpercentile(np.abs(inst), 99):.0f} mm/s")
print(f"forward fraction: {(inst > 0).mean():.2f}   backward fraction: {(inst < 0).mean():.2f}")

## Smoothing sweep

For each `sigma` we measure two things:
- **roughness ratio** = std of the frame-to-frame difference of the smoothed signal, relative to the
  raw instantaneous signal. Lower = spikier high-frequency noise removed.
- **fidelity** = correlation of the smoothed signal with the raw instantaneous signal. Higher = more
  of the original structure retained.

A good `sigma` sits at the elbow: roughness has mostly dropped while fidelity is still high.

In [ ]:
sigmas = [0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0]
raw_rough = np.std(np.diff(inst))

rows = []
smoothed_by_sigma = {}
for s in sigmas:
    sm = gaussian_filter1d(inst, s, mode="nearest")
    smoothed_by_sigma[s] = sm
    rows.append(
        {
            "sigma": s,
            "roughness_ratio": np.std(np.diff(sm)) / raw_rough,
            "fidelity_corr": np.corrcoef(sm, inst)[0, 1],
            "variance_kept": np.var(sm) / np.var(inst),
        }
    )
metrics = pd.DataFrame(rows)
metrics

In [ ]:
CHOSEN_SIGMA = 1.5

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(metrics["sigma"], metrics["roughness_ratio"], "o-", color="tab:red", label="roughness ratio (lower = smoother)")
ax1.set_xlabel("gaussian sigma (frames)")
ax1.set_ylabel("roughness ratio", color="tab:red")
ax1.tick_params(axis="y", labelcolor="tab:red")

ax2 = ax1.twinx()
ax2.plot(metrics["sigma"], metrics["fidelity_corr"], "s-", color="tab:blue", label="fidelity (corr with raw)")
ax2.set_ylabel("fidelity (corr with instantaneous)", color="tab:blue")
ax2.tick_params(axis="y", labelcolor="tab:blue")

ax1.axvline(CHOSEN_SIGMA, color="k", ls="--", alpha=0.6)
ax1.annotate(f"chosen sigma = {CHOSEN_SIGMA}", (CHOSEN_SIGMA, 0.9), xytext=(6, 0),
             textcoords="offset points", va="top")
ax1.set_title("Smoothing trade-off: spike removal vs structure preservation")
fig.tight_layout()
plt.show()

The roughness ratio falls steeply up to ~1.5 and then flattens (diminishing returns), while fidelity
declines steadily. **sigma ≈ 1.5** removes ~80% of the frame-to-frame spikiness while keeping ~0.8
correlation with the raw velocity — a good balance for preserving global structure.

## Visual check on a representative window

In [ ]:
# A 15 s window with clear movement (pick the most active 15 s by |velocity|).
win = int(15 * 60)  # ~15 s at 60 fps
activity = pd.Series(np.abs(inst)).rolling(win, center=True).mean()
c = int(activity.idxmax())
sl = slice(max(0, c - win // 2), c + win // 2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t0[sl], inst[sl], color="0.7", lw=0.8, label="instantaneous")
for s in (1.5, 3.0, 8.0):
    ax.plot(t0[sl], smoothed_by_sigma[s][sl], lw=1.6, label=f"sigma = {s}")
ax.axhline(0, color="k", lw=0.5)
ax.set(xlabel="time (s)", ylabel="signed velocity (mm/s)",
       title="Instantaneous vs Gaussian-smoothed velocity (sigma=8 over-smooths)")
ax.legend()
plt.tight_layout()
plt.show()

## Append the final velocities

Using the tuned `sigma` (the package default), append both columns.

In [ ]:
out = kinematics.append_ear_velocity(
    df, likelihood_threshold=LIKELIHOOD, smoothing_sigma=CHOSEN_SIGMA
)
print([c for c in out.columns if c.startswith("ear_velocity")])
out[["harp_time", "ear_velocity_mm_s", "ear_velocity_smooth_mm_s"]].iloc[1000:1005]